# 01 – Cellpose on an example image

**Purpose:** Test the segmentation workflow (load → inspect → segment → measure) on a
sample image before the Cell Tracking Challenge data are available.
Later notebooks reuse the same steps; only the loading step changes.

**Input:** `skimage.data.human_mitosis()` – 2D fluorescence image of human nuclei
(downloaded once and cached by scikit-image).
**Output:** label image (`masks`) and a per-nucleus table (`df`) – nothing is saved to disk yet.

**Environment:** conda env `bia` (Python 3.12, cellpose 4.2.1.1, napari 0.9.2), MacBook Air M2.
**Date:** 2026-10-30

> Note: There is **no reference annotation** for this image, so segmentation quality can only
> be judged visually here. Quantitative evaluation (IoU against ground truth) follows with the CTC data.

## 1. Imports
Print the numpy version to confirm the notebook runs in the `bia` environment and to record versions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

print("numpy", np.__version__)

## 2. Load and inspect the image
Python equivalent of *Image → Show Info* in Fiji:
- `shape`: image size (rows, cols); time-lapse data will have an extra time axis
- `dtype`: bit depth → possible value range (uint8: 0–255, uint16: 0–65535)
- `min/max`: how much of that range is actually used

**Result (2026-10-30):** shape (512, 512), uint8, min 7 / max 255.

In [ ]:
# Example image: fluorescently labelled human nuclei (HeLa-like, 2D)
img = data.human_mitosis()

print("shape:", img.shape)
print("dtype:", img.dtype)
print("min/max:", img.min(), img.max())

## 3. Check for saturation
max = 255 is the uint8 ceiling, so some pixels might be clipped (detector saturated).
Count them instead of guessing.

**Result:** 2 saturated pixels (0.00 %) → negligible, no action needed.

In [ ]:
# img == 255 gives a boolean image; np.sum counts the True pixels (vectorised, no loop)
n_sat = np.sum(img == 255)
print(f"saturated pixels: {n_sat} ({100 * n_sat / img.size:.2f} %)")

## 4. View in napari
`%gui qt` lets the napari window and the notebook run at the same time (otherwise one blocks the other).
The viewer stays open; later layers are added on top.

In [ ]:
%gui qt
import napari

viewer = napari.Viewer()
viewer.add_image(img, name="nuclei", colormap="gray")

## 5. Segment with Cellpose
- `CellposeModel(gpu=True)`: default pretrained model (Cellpose-SAM). Weights (~1.15 GB) are
  downloaded on first use and cached afterwards.
- `model.eval(img)`: segmentation. Cellpose normalises intensities internally, so the uint8 range
  needs no preprocessing. No diameter is given – Cellpose-SAM is designed to be size-independent.
- `masks` is a **label image**: same size as the input, 0 = background, each nucleus has its own
  integer ID. Therefore `masks.max()` = number of objects. The CTC reference annotations use the same format.

**Result:** GPU (Apple MPS) used; 333 nuclei; masks dtype uint16 (uint8 would allow only 255 labels).
A PyTorch `UserWarning` about sparse tensors appears – internal, harmless.

In [ ]:
from cellpose import models, core

print("GPU available:", core.use_gpu())

model = models.CellposeModel(gpu=True)  # default pretrained model (Cellpose-SAM)
masks, flows, styles = model.eval(img)

print("masks shape:", masks.shape, "| dtype:", masks.dtype)
print("number of nuclei:", masks.max())

## 6. Overlay segmentation in napari
Check visually for three typical error types:
- **merges** – two touching nuclei share one label
- **splits** – one nucleus split into several labels (common during mitosis)
- **misses** – dim nuclei without a label

**Observations (2026-10-30):** touching nuclei are mostly separated well;
one elongated object (probably debris) and several very small objects.

In [ ]:
viewer.add_labels(masks, name="cellpose")

## 7. Measure nucleus size and shape
`regionprops_table` measures each label (similar to *Analyze Particles* in Fiji):
- `area`: size in pixels (no pixel size known for this image → no µm² conversion)
- `eccentricity`: 0 = circle, close to 1 = elongated

The DataFrame (one row per nucleus) is the format that will later be exported as CSV for R.

**Result:** area median 86 px (IQR 69–110), min 16, max 201; eccentricity max 0.97.
The `label` column in `describe()` is just the object ID – its statistics are meaningless.

The small objects (~16 px) and the elongated one (ecc. 0.97) are likely debris or fragments.
Filtering thresholds will be chosen and justified on the CTC data, not on this example.

In [ ]:
import pandas as pd
from skimage.measure import regionprops_table

props = regionprops_table(masks, properties=["label", "area", "eccentricity"])
df = pd.DataFrame(props)

print(df.describe().round(2))

# Histogram doubles as a quality check: tiny objects = fragments, large outliers = merges
plt.hist(df["area"], bins=40)
plt.xlabel("nucleus area (pixels)")
plt.ylabel("count")
plt.show()

## Next steps
- Replace the example image with the CTC dataset Fluo-N2DL-HeLa once the download server is reachable.
- Evaluate segmentation quantitatively against the reference annotations (IoU).